# v2_pinn — función de campo condicional con losses físicas

Wrapper visual del paquete `Models/v2_pinn/`. La red aprende

$$ f(\mathbf{B}_{\text{sens}},\, x, y, z) \;\longrightarrow\; \mathbf{B}(x, y, z) $$

Una FCN recibe los `I·3` valores de sensores + un punto de query xyz y devuelve $B(x,y,z)$ en mT. El loss combina:

- **Datos** — MSE vs `B_grid[i, j]` del simulador (en espacio normalizado).
- **∇·B = 0** y **∇×B = 0** — vía `torch.autograd.grad` sobre las coordenadas, con regla de la cadena explícita para volver a unidades físicas.
- **TV** — $|\nabla B|$ regularización de suavidad.

**Normalización per-componente**: B se normaliza con `(b_mean, b_std)` shape `(3,)` (uno por componente — `By` tiene magnitud y std mucho mayor que `Bx, Bz` por el Halbach horizontal); coordenadas con `(pts_mean, pts_std)` shape `(3,)` (uno por eje — `Nz/Nx` no son simétricos). Esto rompe la equivalencia trivial `∇·B_norm = 0 ⟺ ∇·B_phys = 0`, así que las losses Maxwell aplican `b_std[i]/pts_std[j]` antes de imponer física.

Cada step de SGD ve **una sola configuración** (sample) con K puntos random — necesario para que las losses físicas tengan sentido sobre un campo coherente.

Para cluster headless: `python -m Models.v2_pinn --h5 ... --epochs ...` corre el mismo pipeline sin Jupyter.

In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd() / "PMDKernel" / "Python",
                  Path.cwd().parent / "Python"):
    if (candidate / "Models" / "v2_pinn").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl

from Models.v2_pinn import data
from Models.v2_pinn.model import LitPINN, count_params
from Models.v2_pinn.train import train as pinn_train
from Models.v2_pinn.metrics import evaluate, report

SEED = 42
pl.seed_everything(SEED, workers=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}  |  lightning {pl.__version__}  |  device: {DEVICE}")

Seed set to 42


torch 2.5.1+cu121  |  lightning 2.6.1  |  device: cuda


## 1. Configuración

In [ ]:
REPO_ROOT = Path("..").resolve()
H5_PATH   = REPO_ROOT / "data" / "datasets" / "v1_xy100_z225_step10_n5000.h5"

VAL_FRAC  = 0.15
TEST_FRAC = 0.15

# Arquitectura PINN: capas crecientes, activación SiLU
HIDDEN_LAYERS  = [256, 256, 256, 256]
ACTIVATION     = "silu"

# Pesos de las losses físicas (data tiene peso 1.0 implícito).
# Solo se usan cuando BALANCE_GRADS=False — en modo balanceado se ignoran y
# las escalas se calculan dinámicamente en cada step.
LAMBDA_DIV = 1e-2
LAMBDA_ROT = 1e-2
LAMBDA_TV  = 1e-4

# Gradient balancing: rescala dinámicamente las losses físicas para igualar la
# norma del gradiente con la de `loss_data` en cada paso. Tradeoff: 3 backwards
# adicionales por step (~30-50% más lento), pero elimina la búsqueda manual de λ.
BALANCE_GRADS = True

# Batching: cada step toma 1 sample y K puntos random de su grilla
POINTS_PER_SAMPLE = 4096

N_EPOCHS      = 200
LR            = 1e-3
WEIGHT_DECAY  = 1e-5
PATIENCE      = 30
GRAD_CLIP     = 1.0   # PINN tiende a tener grads grandes al inicio.
                       # En modo balanceado se aplica vía `manual_clip_val`.

NUM_WORKERS = 0       # Windows local: 0; cluster Linux: 4-8 + pin_memory=True
PIN_MEMORY  = False

RUN_TAG = f"v2_pinn_{H5_PATH.stem}{'_balanced' if BALANCE_GRADS else ''}"
OUT_DIR = REPO_ROOT / "python" / "Models" / "v2_pinn" / "logs" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"H5_PATH = {H5_PATH}")
print(f"RUN_TAG = {RUN_TAG}")
print(f"OUT_DIR = {OUT_DIR}")

## 2. Carga de metadatos

In [3]:
ds = data.load_dataset(H5_PATH)

print(f"Muestras (N)   : {ds['N']}")
print(f"Sensores (I)   : {ds['I']}  → entrada al modelo: I*3 + 3 = {ds['I']*3 + 3}")
print(f"Grilla (J)     : {ds['Nx']} x {ds['Ny']} x {ds['Nz']} = {ds['J']}")
print(f"Rango x        : [{ds['grid_x'].min():.0f}, {ds['grid_x'].max():.0f}] mm")
print(f"Rango y        : [{ds['grid_y'].min():.0f}, {ds['grid_y'].max():.0f}] mm")
print(f"Rango z        : [{ds['grid_z'].min():.0f}, {ds['grid_z'].max():.0f}] mm")
print(f"Perturb        : kind={ds['attrs'].get('kind')}, sigma_deg={ds['attrs'].get('sigma_deg')}")

Muestras (N)   : 5000
Sensores (I)   : 180  → entrada al modelo: I*3 + 3 = 543
Grilla (J)     : 21 x 21 x 46 = 20286
Rango x        : [-100, 100] mm
Rango y        : [-100, 100] mm
Rango z        : [-225, 225] mm
Perturb        : kind=b'both', sigma_deg=1.0


## 3. Split + scalers + DataLoaders

Normalización per-componente:

- **Sensores** (input): `StandardScaler` per-feature → media 0, std 1 por feature.
- **Coordenadas xyz** (input): `(xyz - pts_mean) / pts_std` con vector `(3,)` independiente por eje. `pts_std[2]` es típicamente 2-3× mayor que `pts_std[0,1]` (Nz > Nx,Ny).
- **B target**: `(B - b_mean) / b_std` con vector `(3,)` independiente por componente. `b_std[1]` (By) es ~10-20× mayor que `b_std[0,2]` (Bx, Bz) por el Halbach horizontal.

Las stats viajan en el `.ckpt` como buffers del modelo → la inferencia desnormaliza sin metadatos externos.

In [4]:
splits = data.split_indices(ds["N"], val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED)
stats  = data.compute_train_stats(H5_PATH, splits["train"], chunk=64)
x_scaler = stats["x_scaler"]
b_mean   = stats["b_mean"]
b_std    = stats["b_std"]
pts_mean = stats["pts_mean"]
pts_std  = stats["pts_std"]

loader_kwargs = dict(points_per_sample=POINTS_PER_SAMPLE,
                     x_scaler=x_scaler,
                     b_mean=b_mean, b_std=b_std,
                     pts_mean=pts_mean, pts_std=pts_std,
                     num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

loader_tr = data.make_loader(H5_PATH, splits["train"], shuffle=True,  seed=SEED,     **loader_kwargs)
loader_va = data.make_loader(H5_PATH, splits["val"],   shuffle=False, seed=SEED+1,   **loader_kwargs)

print(f"train={splits['n_train']}   val={splits['n_val']}   test={splits['n_test']}")
print(f"x_scaler:    mean range [{x_scaler.mean_.min():.3e}, {x_scaler.mean_.max():.3e}]")
print(f"             scale range [{x_scaler.scale_.min():.3e}, {x_scaler.scale_.max():.3e}]")
print(f"b_mean   = [{b_mean[0]:+.4f}, {b_mean[1]:+.4f}, {b_mean[2]:+.4f}] mT  (Bx, By, Bz)")
print(f"b_std    = [{b_std[0]:.4f},  {b_std[1]:.4f},  {b_std[2]:.4f}] mT")
print(f"pts_mean = [{pts_mean[0]:+.2f}, {pts_mean[1]:+.2f}, {pts_mean[2]:+.2f}] mm")
print(f"pts_std  = [{pts_std[0]:.2f}, {pts_std[1]:.2f}, {pts_std[2]:.2f}] mm")
print(f"K (puntos por step): {POINTS_PER_SAMPLE}")

sensors_b, points_b, b_b = next(iter(loader_tr))
print(f"\nbatch shapes: sensors={tuple(sensors_b.shape)}, points={tuple(points_b.shape)}, B={tuple(b_b.shape)}")
print(f"sensors range: [{sensors_b.min():.3f}, {sensors_b.max():.3f}]   (z-score)")
print(f"points range : [{points_b.min():.3f}, {points_b.max():.3f}]    ((xyz - mean)/std)")
print(f"B range      : [{b_b.min():.3f}, {b_b.max():.3f}]    ((B - b_mean)/b_std)")

train=3500   val=750   test=750
x_scaler:    mean range [-1.933e+01, 3.067e+01]
             scale range [1.053e-01, 5.345e-01]
b_mean   = [+0.0009, -50.7924, -0.0003] mT  (Bx, By, Bz)
b_std    = [4.5637,  3.7506,  9.3070] mT
pts_mean = [+0.00, +0.00, +0.00] mm
pts_std  = [60.55, 60.55, 132.76] mm
K (puntos por step): 4096

batch shapes: sensors=(4096, 540), points=(4096, 3), B=(4096, 3)
sensors range: [-2.761, 3.152]   (z-score)
points range : [-1.695, 1.695]    ((xyz - mean)/std)
B range      : [-8.705, 10.541]    ((B - b_mean)/b_std)


## 4. Modelo PINN

In [5]:
lit_model = LitPINN(
    n_sensors=ds["I"], hidden_layers=HIDDEN_LAYERS,
    activation=ACTIVATION,
    lr=LR, weight_decay=WEIGHT_DECAY,
    lambda_div=LAMBDA_DIV, lambda_rot=LAMBDA_ROT, lambda_tv=LAMBDA_TV,
    balance_grads=BALANCE_GRADS,
    manual_clip_val=(GRAD_CLIP if BALANCE_GRADS else None),
    b_mean=tuple(b_mean.tolist()), b_std=tuple(b_std.tolist()),
    pts_mean=tuple(pts_mean.tolist()), pts_std=tuple(pts_std.tolist()),
)
print(lit_model)
print(f"\nParámetros entrenables: {count_params(lit_model):,}")
if BALANCE_GRADS:
    print("Modo: gradient balancing (λ ignorados; clip manual)")
else:
    print(f"Modo: λ fijos — λ_div={LAMBDA_DIV}  λ_rot={LAMBDA_ROT}  λ_TV={LAMBDA_TV}")

LitPINN(
  (net): PINN(
    (net): Sequential(
      (0): Linear(in_features=543, out_features=256, bias=True)
      (1): SiLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): SiLU()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): SiLU()
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): SiLU()
      (8): Linear(in_features=256, out_features=3, bias=True)
    )
  )
  (train_mse): MeanSquaredError()
  (val_mse): MeanSquaredError()
)

Parámetros entrenables: 337,411
Modo: gradient balancing (λ ignorados; clip manual)


## 5. Entrenamiento

Con `gradient_clip_val=1.0` para domar los grads autograd-derivados que pueden
explotar al inicio de PINN (las físicas dependen de derivadas de la salida).

In [ ]:
trainer = pinn_train(
    lit_model, loader_tr, loader_va,
    n_epochs=N_EPOCHS, patience=PATIENCE,
    ckpt_dir=OUT_DIR, run_tag=RUN_TAG,
    gradient_clip_val=(None if BALANCE_GRADS else GRAD_CLIP),
)
best_ckpt_path = trainer.checkpoint_callback.best_model_path
comet_exp      = trainer.logger.experiment
comet_url      = getattr(comet_exp, "url", None)
comet_key      = comet_exp.id

print(f"\nMejor checkpoint : {best_ckpt_path}")
print(f"Comet experiment : {comet_key}")
if comet_url:
    print(f"Comet URL        : {comet_url}")

lit_model = LitPINN.load_from_checkpoint(best_ckpt_path)

The parameter `experiment_name` is deprecated, please use `name` instead.
The parameter `project_name` is deprecated, please use `project` instead.
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/sebasti-n-vallejos/pmdkernel/069132ade776454fa4d709059b0239e1

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torc

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net       │ PINN             │  337 K │ train │     0 │
│ 1 │ train_mse │ MeanSquaredError │      0 │ train │     0 │
│ 2 │ val_mse   │ MeanSquaredError │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 337 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 337 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Poney\Desktop\Ipre\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\Poney\Desktop\Ipre\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

In [ ]:
# Curvas via Comet API (en vez de CSV local). `comet_ml.API` consulta el
# experiment recién corrido por su key y devuelve las series de cada métrica.
# Lee credenciales de `~/.comet.config` (mismo source que el training).
from comet_ml import API

api = API()
api_exp = api.get_experiment_by_key(comet_key)

def _series(name):
    """Devuelve (epochs, values) de una métrica, o (None, None) si no existe."""
    raw = api_exp.get_metrics(name)   # list of dicts {step, epoch, metricValue, ...}
    if not raw:
        return None, None
    rows = []
    for r in raw:
        epoch = r.get("epoch")
        val   = r.get("metricValue")
        if epoch is None or val is None:
            continue
        rows.append((int(epoch), float(val)))
    if not rows:
        return None, None
    df_m = (pd.DataFrame(rows, columns=["epoch", "value"])
              .groupby("epoch", as_index=False).first()
              .sort_values("epoch"))
    return df_m["epoch"].to_numpy(), df_m["value"].to_numpy()

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)

def _plot(ax, t_name, v_name, title, log=True):
    e_t, y_t = _series(t_name)
    e_v, y_v = _series(v_name)
    if y_t is not None: ax.plot(e_t, y_t, label="train")
    if y_v is not None: ax.plot(e_v, y_v, label="val")
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    if log: ax.set_yscale("log")

_plot(axes[0, 0], "train_loss", "val_loss", "loss total")
_plot(axes[0, 1], "train_data", "val_data", "loss data (norm)")
_plot(axes[0, 2], "train_div",  "val_div",  "loss ∇·B  (mT/mm)²")
_plot(axes[1, 0], "train_rot",  "val_rot",  "loss ∇×B  (mT/mm)²")
_plot(axes[1, 1], "train_tv",   "val_tv",   "loss TV  (mT/mm)")

ax = axes[1, 2]
e_d, y_d = _series("train_alpha_div")
if y_d is not None:
    e_r, y_r = _series("train_alpha_rot")
    e_v, y_v = _series("train_alpha_tv")
    ax.plot(e_d, y_d, label="α_div")
    if y_r is not None: ax.plot(e_r, y_r, label="α_rot")
    if y_v is not None: ax.plot(e_v, y_v, label="α_tv")
    ax.set_yscale("log"); ax.set_title("escalas balanceadas (α_k)")
    ax.legend(); ax.grid(alpha=0.3)
else:
    ax.set_title("(λ fijos — sin α)"); ax.axis("off")

axes[1, 0].set_xlabel("epoch"); axes[1, 1].set_xlabel("epoch"); axes[1, 2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 6. Evaluación final

Para reportar RMSE/R² comparable a v1, evaluamos cada sample del split en
**todos los J puntos** de su grilla (no solo K aleatorios). Esto pasa por
`evaluate(...)` que itera sobre samples y predice todo el volumen.

In [ ]:
eval_kwargs = dict(
    x_scaler=x_scaler,
    b_mean=b_mean, b_std=b_std,
    pts_mean=pts_mean, pts_std=pts_std,
    device=DEVICE, rmse_per_component=True,
)

m_tr = evaluate(lit_model, H5_PATH, splits["train"], **eval_kwargs)
m_va = evaluate(lit_model, H5_PATH, splits["val"],   **eval_kwargs)
m_te = evaluate(lit_model, H5_PATH, splits["test"],  **eval_kwargs)

report("train", m_tr)
report("val",   m_va)
report("test",  m_te)

## 7. Guardar scalers + metadata

El `ModelCheckpoint` ya guardó el `.ckpt`. Acá persistimos lo que Lightning no
guarda: scaler de sensores (StandardScaler fitted), índices del split, y las
métricas finales para tener todo en un mismo lugar.

In [ ]:
aux_path = OUT_DIR / f"{RUN_TAG}_aux.pt"
torch.save({
    "x_scaler":    x_scaler,
    "x_mean":      x_scaler.mean_.astype(np.float32),
    "x_scale":     x_scaler.scale_.astype(np.float32),
    "b_mean":      b_mean.astype(np.float32),
    "b_std":       b_std.astype(np.float32),
    "pts_mean":    pts_mean.astype(np.float32),
    "pts_std":     pts_std.astype(np.float32),
    "splits":      splits,
    "metrics":     {"train": m_tr, "val": m_va, "test": m_te},
    "hparams": {
        "hidden_layers":     HIDDEN_LAYERS,
        "activation":        ACTIVATION,
        "lambda_div":        LAMBDA_DIV,
        "lambda_rot":        LAMBDA_ROT,
        "lambda_tv":         LAMBDA_TV,
        "balance_grads":     BALANCE_GRADS,
        "grad_clip":         GRAD_CLIP,
        "points_per_sample": POINTS_PER_SAMPLE,
        "lr":                LR,
        "weight_decay":      WEIGHT_DECAY,
        "n_epochs":          N_EPOCHS,
        "patience":          PATIENCE,
    },
    "h5_path":   str(H5_PATH),
    "ckpt_path": best_ckpt_path,
}, aux_path)
print(f"Best checkpoint Lightning : {best_ckpt_path}")
print(f"Scalers + metadata        : {aux_path}")